# Exploratory 2 — EDA & Maps (Data Bersih)

Exploratory Data Analysis and map visualizations using cleaned village-level data (BPS + PODES + IUP flag) from **Data/DATA_BERSIH**.

Written by: Atha Bintang Wahyu M.

In [ ]:
# Setup: paths and imports
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

BASE = Path("/Users/athamawardi/Desktop/Research-Projects/KRE_Equity")
DATA_BERSIH = BASE / "Data" / "DATA_BERSIH"
adm4_files = sorted(DATA_BERSIH.glob("adm4_podes_*_with_iup_flag.gpkg"))
print("Available adm4 files:", [f.name for f in adm4_files])


In [ ]:
# Load 2024 village-level data (primary EDA); optionally load another year for panel
gdf = gpd.read_file(DATA_BERSIH / "adm4_podes_2024_with_iup_flag.gpkg")
print("Shape:", gdf.shape)
print("CRS:", gdf.crs)
gdf.head(2)


## 1. Data structure and missing values

In [ ]:
# Basic info
print(gdf.info())
print("\n--- Missing (top 20) ---")
miss = gdf.isnull().sum().sort_values(ascending=False)
print(miss[miss > 0].head(20))


### Key columns (IUP & identifiers)

In [ ]:
key_cols = ["id_desa_numeric", "NAMA_PROV", "NAMA_KAB", "NAMA_KEC", "NAMA_DESA", "has_iup",
         "n_wiup", "area_hectares_iup", "pct_iup_coverage"]
key = [c for c in key_cols if c in gdf.columns]
gdf[key].describe(include="all")


## 2. Target variable: has_iup

In [ ]:
# Villages with vs without IUP
print(gdf["has_iup"].value_counts().to_string())
print("\nShare with IUP: {:.2%}".format(gdf["has_iup"].mean()))


## 3. Numeric EDA — IUP aggregates (villages with IUP)

In [ ]:
# Among villages with IUP
with_iup = gdf[gdf["has_iup"] == 1]
for col in ["n_wiup", "area_hectares_iup", "pct_iup_coverage"]:
    if col not in with_iup.columns:
        continue
    print(f"--- {col} ---")
    print(with_iup[col].describe().to_string())
    print()


## 4. Categorical EDA — Province and district

In [ ]:
# Count by province
prov_counts = gdf["NAMA_PROV"].value_counts()
print("Top 15 provinces (all villages):")
print(prov_counts.head(15).to_string())
print("\nVillages with IUP by province:")
iup_by_prov = gdf.groupby("NAMA_PROV")["has_iup"].agg(["sum", "count"]).assign(pct=lambda x: x["sum"]/x["count"]*100)
iup_by_prov = iup_by_prov.sort_values("sum", ascending=False)
print(iup_by_prov.head(15).to_string())


## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
with_iup = gdf[gdf["has_iup"] == 1]

# 1) has_iup bar
gdf["has_iup"].value_counts().sort_index().plot(kind="bar", ax=axes[0,0], color=["#2ecc71","#3498db"])
axes[0,0].set_title("Villages: with IUP vs without")
axes[0,0].set_xticklabels(["No IUP", "Has IUP"], rotation=0)

# 2) IUP share by province (top 15)
iup_by_prov.head(15)["sum"].plot(kind="barh", ax=axes[0,1], color="steelblue")
axes[0,1].set_title("Number of villages with IUP (top 15 provinces)")

# 3) Distribution of IUP area (has_iup==1)
if "area_hectares_iup" in gdf.columns and len(with_iup) > 0:
    with_iup["area_hectares_iup"].clip(upper=with_iup["area_hectares_iup"].quantile(0.95)).hist(ax=axes[1,0], bins=50, edgecolor="white")
    axes[1,0].set_title("Area (ha) under IUP (villages with IUP, 95% quantile cap)")

# 4) Distribution of pct_iup_coverage
if "pct_iup_coverage" in gdf.columns and len(with_iup) > 0:
    with_iup["pct_iup_coverage"].hist(ax=axes[1,1], bins=50, edgecolor="white", color="coral", alpha=0.8)
    axes[1,1].set_title("IUP coverage (%) within village")

plt.tight_layout()
plt.show()


## 6. Maps

In [ ]:
# Simplify geometry for faster plotting (optional)
gdf_plot = gdf.copy()
if gdf_plot.crs is None:
    gdf_plot.set_crs(4326, inplace=True)

fig, ax = plt.subplots(1, 1, figsize=(14, 10))
gdf_plot.plot(column="has_iup", categorical=True, legend=True, ax=ax, cmap="RdYlGn_r", linewidth=0.1, edgecolor="gray")
ax.set_title("Villages: Has IUP (1) vs No IUP (0) — 2024")
ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Choropleth: share of villages with IUP by province
prov_agg = gdf.groupby("NAMA_PROV").agg(
    has_iup_sum=("has_iup", "sum"),
    total=("has_iup", "count")
).reset_index()
prov_agg["pct_iup"] = prov_agg["has_iup_sum"] / prov_agg["total"] * 100
# Dissolve to province and join
gdf_prov = gdf.dissolve(by="NAMA_PROV", aggfunc="first").reset_index()
gdf_prov = gdf_prov.merge(prov_agg[["NAMA_PROV", "pct_iup", "has_iup_sum"]], on="NAMA_PROV", how="left")

fig, ax = plt.subplots(1, 1, figsize=(14, 10))
gdf_prov.plot(column="pct_iup", legend=True, ax=ax, cmap="YlOrRd", edgecolor="black", linewidth=0.3)
ax.set_title("% of villages with IUP by province (2024)")
ax.axis("off")
plt.tight_layout()
plt.show()


## 7. Panel snapshot: villages with IUP by year


In [ ]:
# Quick comparison across years (one row per year)
years = [2011, 2014, 2018, 2019, 2020, 2021, 2024]
rows = []
for y in years:
    p = DATA_BERSIH / f"adm4_podes_{y}_with_iup_flag.gpkg"
    if not p.exists():
        continue
    g = gpd.read_file(p)
    if "has_iup" not in g.columns:
        continue
    rows.append({"year": y, "total": len(g), "with_iup": g["has_iup"].sum(), "pct": g["has_iup"].mean() * 100})
panel = pd.DataFrame(rows)
print(panel.to_string())
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.bar(panel["year"], panel["with_iup"], color="steelblue", edgecolor="black")
ax.set_xlabel("Year")
ax.set_ylabel("Villages with IUP")
ax.set_title("Number of villages with IUP by year (data bersih)")
plt.tight_layout()
plt.show()


In [ ]:
# Map: IUP area (ha) — villages with IUP only, log scale for visibility
with_iup_gdf = gdf[gdf["has_iup"] == 1].copy()
with_iup_gdf["area_log"] = np.log1p(with_iup_gdf["area_hectares_iup"].fillna(0))

fig, ax = plt.subplots(1, 1, figsize=(14, 10))
with_iup_gdf.plot(column="area_log", legend=True, ax=ax, cmap="viridis", linewidth=0.05, edgecolor="gray")
ax.set_title("Villages with IUP: log(1 + area_hectares_iup)")
ax.axis("off")
plt.tight_layout()
plt.show()
